# Regression and Classification Workflow


In [1]:
import numpy as np

from pynns import (
    dy_d,
    dy_dx,
    nns_boost,
    nns_diff,
    nns_m_reg,
    nns_norm,
    nns_part,
    nns_reg,
    nns_stack,
    prepare_factor_predictors,
)

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(7)


## Training data


In [2]:
n = 120
age = rng.integers(22, 68, size=n).astype(float)
income = rng.normal(72.0, 14.0, size=n)
activity = rng.uniform(0.0, 1.0, size=n)
plan = np.where(activity > 0.68, "pro", np.where(income < 67.0, "basic", "plus"))
plan_levels = ("basic", "plus", "pro")

spend = (
    18.0
    + 0.72 * income
    - 0.12 * age
    + 22.0 * np.sin(np.pi * activity)
    + np.where(plan == "pro", 24.0, np.where(plan == "plus", 10.0, 0.0))
    + rng.normal(0.0, 4.0, size=n)
)

raw_x = np.empty((n, 4), dtype=object)
raw_x[:, 0] = income
raw_x[:, 1] = activity
raw_x[:, 2] = age
raw_x[:, 3] = plan

new_customers = np.array(
    [
        [82.0, 0.72, 38.0, "pro"],
        [58.0, 0.20, 55.0, "basic"],
        [70.0, 0.50, 44.0, "plus"],
    ],
    dtype=object,
)

print("first rows of raw predictors:")
print(raw_x[:5])
print("spend range:", round(float(spend.min()), 2), "to", round(float(spend.max()), 2))


first rows of raw predictors:
[[62.54072848592086 0.8776913666495335 65.0 'pro']
 [74.84394054545453 0.5233041529751773 50.0 'plus']
 [65.51369392846219 0.9156354351007324 53.0 'pro']
 [73.78175775716163 0.04665223795388718 63.0 'plus']
 [55.379276610098046 0.030288833931601977 48.0 'basic']]
spend range: 52.89 to 122.2


## Univariate nonlinear regression


In [3]:
activity_points = np.array([0.10, 0.50, 0.90], dtype=np.float64)
one_feature = nns_reg(
    activity,
    spend,
    point_est=activity_points,
    confidence_interval=None,
    noise_reduction="median",
)
print("R2:", round(float(one_feature["R2"]), 4))
print("activity points:", activity_points)
print("predicted spend:", np.round(one_feature["Point.est"], 2))
print("first fitted rows: x, y, y.hat, gradient")
fitted = one_feature["Fitted.xy"]
print(np.column_stack((fitted["x"][:5], fitted["y"][:5], fitted["y.hat"][:5], fitted["gradient"][:5])))


R2: 0.3934
activity points: [0.1 0.5 0.9]
predicted spend: [78.1  98.8  94.02]
first fitted rows: x, y, y.hat, gradient
[[   0.8777   86.6618   93.5385  -29.1402]
 [   0.5233   98.0459   98.0459 -706.5967]
 [   0.9156   94.62     95.1022   68.9965]
 [   0.0467   78.9985   77.4832 -426.5899]
 [   0.0303   53.9695   63.0654 1234.707 ]]


## Partition map


In [4]:
part = nns_part(activity, spend, order=3, obs_req=8, type="XONLY", noise_reduction="median")
print("selected order:", part["order"])
print("first 12 quadrant ids:", part["dt"]["quadrant"][:12])
print("regression points: x, y")
rp = part["regression.points"]
print(np.column_stack((rp["x"], rp["y"])))


selected order: 3
first 12 quadrant ids: ['q222' 'q122' 'q222' 'q111' 'q111' 'q111' 'q121' 'q121' 'q112' 'q211'
 'q111' 'q211']
regression points: x, y
[[ 0.1069 77.3254]
 [ 0.3285 95.0548]
 [ 0.6429 93.6009]
 [ 0.8485 96.2964]]


## Factor encoding


In [5]:
design = prepare_factor_predictors(
    raw_x,
    point_est=new_customers,
    factor_levels=[None, None, None, plan_levels],
    names=["income", "activity", "age", "plan"],
)
print("feature names:", design.feature_names)
print("design shape:", design.x.shape)
print("new-customer design rows:")
print(design.point_est)


feature names: ('income', 'activity', 'age', 'plan_basic', 'plan_plus', 'plan_pro')
design shape: (120, 6)
new-customer design rows:
[[82.    0.72 38.    0.    0.    1.  ]
 [58.    0.2  55.    1.    0.    0.  ]
 [70.    0.5  44.    0.    1.    0.  ]]


## Multivariate regression and stacking


In [6]:
mreg = nns_m_reg(
    design.x,
    spend,
    point_est=design.point_est,
    n_best=3,
    confidence_interval=None,
)
stacked = nns_stack(
    design.x,
    spend,
    design.point_est,
    method=(1, 2),
    folds=2,
    cv_size=0.25,
    pred_int=None,
    random_seed=11,
)
print("nns_m_reg R2:", round(float(mreg["R2"]), 4))
print("nns_m_reg point estimates:", np.round(mreg["Point.est"], 2))
print("stacked point estimates:", np.round(stacked["stack"], 2))
print("stack parameters: n_best=", stacked["NNS.reg.n.best"], "threshold=", round(float(stacked["NNS.dim.red.threshold"]), 4))


nns_m_reg R2: 0.9833
nns_m_reg point estimates: [112.94  55.37  90.91]
stacked point estimates: [100.17  59.37  93.94]
stack parameters: n_best= 1.0 threshold= 0.37


## Classification


In [7]:
state = np.where((activity < 0.25) & (plan == "basic"), 3.0, np.where(spend > 92.0, 1.0, 2.0))
class_model = nns_m_reg(
    design.x,
    state,
    type="class",
    point_est=design.point_est,
    n_best=1,
    confidence_interval=None,
)
class_stack = nns_stack(
    design.x,
    state,
    design.point_est,
    type="class",
    method=(1, 2),
    folds=1,
    cv_size=0.25,
    pred_int=None,
    random_seed=3,
)
print("class counts:", {int(label): int(np.sum(state == label)) for label in np.unique(state)})
print("nns_m_reg class accuracy proxy:", round(float(class_model["R2"]), 4))
print("class_model predictions:", class_model["Point.est"])
print("class_stack predictions:", class_stack["stack"])


class counts: {1: 55, 2: 51, 3: 14}
nns_m_reg class accuracy proxy: 1.0
class_model predictions: [1. 3. 2.]
class_stack predictions: [1. 2. 1.]


## Diagnostics


In [8]:
small_x = design.x[:, :4]
small_points = design.point_est[:, :4]
boost = nns_boost(
    small_x,
    spend,
    small_points,
    learner_trials=8,
    epochs=2,
    random_seed=5,
    pred_int=None,
    feature_importance=True,
)
overall_activity_slope = dy_dx(activity, spend, eval_point="overall")
local_partials = dy_d(
    np.column_stack((income, activity, age)),
    spend,
    wrt=np.array([1, 2, 3]),
    eval_points=np.mean(np.column_stack((income, activity, age)), axis=0),
)
derivative = nns_diff(lambda z: z**3 + 2.0 * z, 1.5)
normalized = nns_norm(np.column_stack((income, activity * 100.0, age)), linear=True)

print("boost keys:", sorted(boost.keys()))
print("boost predictions:", np.round(boost["results"], 2))
print("overall dy/dactivity:", round(float(overall_activity_slope), 4))
print("local partial derivatives:", {key: np.round(value, 4) for key, value in local_partials.items()})
print("nns_diff derivative for z^3 + 2z at 1.5:", derivative["DERIVATIVE"])
print("normalized column means:", np.round(np.mean(normalized, axis=0), 4))


boost keys: ['feature.frequency', 'feature.weights', 'n.best', 'pred.int', 'results']
boost predictions: [102.88  79.01  90.66]
overall dy/dactivity: 2.657
local partial derivatives: {'First': array([[0.1595, 0.3088, 0.1823],
       [0.1653, 0.8723, 0.1475],
       [0.2073, 1.7184, 0.1591]]), 'Second': array([[-0.0016,  0.217 ,  0.0021],
       [ 0.0027, -1.1047,  0.0017],
       [-0.0048, -2.9808,  0.0013]])}
nns_diff derivative for z^3 + 2z at 1.5: 8.750000003672
normalized column means: [54.6659 54.6659 54.6659]
